<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fengyee/TorchLeet/blob/main/v4/ml-fundamentals/decision-tree/decision-tree.ipynb)

Check your work: `pip install -e python/` once from the repo root (on Colab: `!pip install -q "git+https://github.com/Fengyee/TorchLeet#subdirectory=python"`), then `from torchleet import check; check("decision-tree", ...)`


# Decision Tree Classifier (CART)

**Difficulty**: 🟡 Medium

---

### Problem Statement

Decision trees are the weak learners inside random forests and gradient-boosted trees
(XGBoost, LightGBM, CatBoost), which are still the default models for tabular data. They also
come up constantly in ML-breadth interviews. **CART** (Breiman et al., 1984) grows a binary tree
greedily. At every node it tries each feature $f$ and threshold $t$, sends samples with
$x_f \le t$ left and the rest right, and keeps the split whose children are purest.

For a node whose samples have class proportions $p = (p_1, \dots, p_C)$, the **Gini impurity** is

$$G(p) = 1 - \sum_{c=1}^{C} p_c^2 .$$

A split of $n$ samples into $n_L$ left and $n_R$ right is scored by the **weighted child impurity**

$$\frac{n_L}{n}\,G(p_L) + \frac{n_R}{n}\,G(p_R),$$

and CART picks the split that minimizes it. Writing that loop is easy. What interviewers probe
is the **split search**: recomputing the impurity from scratch for every threshold costs
$O(n^2)$ per feature. Sorting the column once and using cumulative class counts scores *every*
threshold in one vectorized pass, in $O(n \log n)$.

---

### Requirements

1. **`DecisionTreeClassifier(max_depth=None, min_samples_split=2)`**
   - **`fit(X, y)`**: `X` is a `(N, D)` float tensor and `y` a `(N,)` integer tensor of labels in
     `[0, C)` with **`C = y.max() + 1`**. Returns `self`.
   - **`predict_proba(X)`** → `(M, C)`: the class distribution stored in the leaf each row falls into.
   - **`predict(X)`** → `(M,)`: the most probable class (argmax of `predict_proba`).
   - **`get_depth()`** → `int`: depth of the fitted tree. The root is depth 0, so a tree that is a
     single leaf has depth 0.
2. **Candidate thresholds** for feature $f$ at a node are the **midpoints between consecutive
   distinct sorted values** of $x_f$ among that node's samples. A row goes **left iff
   `x[f] <= threshold`**. Never cut between two equal values.
3. **Split choice**: the (feature, threshold) with the lowest weighted child Gini over all features.
   Ties may be broken any way.
4. **Stopping**: a node becomes a leaf if
   - its depth equals `max_depth` (`None` means unlimited), or
   - it has fewer than `min_samples_split` samples, or
   - no candidate split gives a weighted child impurity **strictly lower** than the node's own
     impurity. This covers pure nodes and nodes whose rows are all identical.
5. **Leaves** store the **class distribution**: the fraction of the node's training samples in each
   of the `C` classes, not just the majority class.

---

### Constraints

- ✅ PyTorch tensor ops. Python loops over *nodes* are fine.
- ✅ Score all thresholds of a feature with one sort + cumulative counts, not a loop over thresholds.
- ❌ No scikit-learn or other ML libraries.
- ❌ Don't recompute the impurity from scratch for each candidate threshold ($O(n^2)$ per feature).
  It gives the right answer, but it's the version an interviewer pushes back on.

---

<details>
  <summary>💡 Hint</summary>

  - Sort the node's samples by one feature. A cut between sorted positions $i$ and $i+1$ sends
    exactly the first $i+1$ samples left. How do the class counts on the left change as $i$ grows?
  - Turn the labels into a one-hot matrix `(n, C)`. Its cumulative sum along the sorted order gives
    the left counts for *every* cut at once, and right = total − left.
  - With counts $c$ (and $n = \sum_c c_c$), $n \cdot G = n - \lVert c\rVert^2 / n$. Rewrite the
    weighted child impurity with this identity, then pick the cut with an `argmax`/`argmin`.
  - `X.sort(dim=0)` sorts every column at once, so you can score all features in a single pass.
  - Positions where two neighbouring sorted values are equal are not real cuts. Mask them out.
  - Recursion is natural for growing the tree. For predicting, think about how to move *all* rows
    down one level at a time.

</details>

In [ ]:
import math

import torch
import torch.nn.functional as F

In [ ]:
# Three overlapping Gaussian blobs in 2D: a 3-class problem no single split can solve.
torch.manual_seed(0)
centers = torch.tensor([[0.0, 0.0], [2.5, 0.5], [1.0, 2.5]])
X = torch.cat([c + 0.8 * torch.randn(70, 2) for c in centers])     # (210, 2)
y = torch.arange(3).repeat_interleave(70)                           # (210,) labels 0, 1, 2
perm = torch.randperm(len(y))
X, y = X[perm], y[perm]
X_train, y_train, X_test, y_test = X[:150], y[:150], X[150:], y[150:]
print("X_train", tuple(X_train.shape), "class counts", y_train.bincount().tolist())

In [ ]:
class DecisionTreeClassifier:
    """CART classification tree: binary axis-aligned splits chosen by Gini impurity."""

    def __init__(self, max_depth=None, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split

    def fit(self, X, y):
        """
        Args:
            X: (N, D) float features.
            y: (N,) integer class labels in [0, C), with C = y.max() + 1.
        Returns:
            self
        """
        # TODO: grow the tree from the root. At each node: stop (max_depth, min_samples_split,
        #       no strict impurity decrease) or find the best split and grow both children.
        raise NotImplementedError

    def get_depth(self):
        """Depth of the fitted tree; a tree that is a single leaf has depth 0."""
        # TODO
        raise NotImplementedError

    def predict_proba(self, X):
        """
        Args:
            X: (M, D) float features.
        Returns:
            (M, C) class distribution of the leaf each row lands in.
        """
        # TODO
        raise NotImplementedError

    def predict(self, X):
        """(M, D) -> (M,) most probable class."""
        # TODO
        raise NotImplementedError


def best_gini_split(X, Y):
    """(Suggested helper) best split of one node, or None if no split lowers the impurity.

    Args:
        X: (n, D) features of the samples at this node.
        Y: (n, C) one-hot labels of the same samples.
    Returns:
        (feature, threshold) or None.
    """
    # TODO: sort each column, cumsum the one-hot labels in sorted order, score every cut,
    #       mask cuts between equal values, take the best, return the midpoint threshold.
    raise NotImplementedError

In [ ]:
# Test 1: a depth-1 tree is exactly the best single split, found here by brute force
def gini(labels, C):
    p = torch.bincount(labels, minlength=C).double() / len(labels)
    return 1 - (p ** 2).sum().item()

def brute_force_stump(X, y, C):
    best = (float("inf"), None, None)
    for f in range(X.shape[1]):
        vals = torch.unique(X[:, f])                          # sorted distinct values
        for t in ((vals[:-1] + vals[1:]) / 2).tolist():       # midpoints
            m = X[:, f] <= t
            wg = (m.sum().item() * gini(y[m], C) + (~m).sum().item() * gini(y[~m], C)) / len(y)
            if wg < best[0]:
                best = (wg, f, t)
    return best

best_wg, f, t = brute_force_stump(X_train, y_train, 3)
stump = DecisionTreeClassifier(max_depth=1).fit(X_train, y_train)
P = stump.predict_proba(X_train)
left = X_train[:, f] <= t
for side in (left, ~left):
    want = torch.bincount(y_train[side], minlength=3).float() / side.sum()
    assert torch.allclose(P[side], want.expand(int(side.sum()), 3), atol=1e-6), \
        "the depth-1 tree's leaves differ from the brute-force best stump"
print(f"best stump: feature {f}, threshold {t:.4f}, weighted Gini {best_wg:.4f}")
print("Test 1 passed: depth-1 tree == brute-force best stump")

In [ ]:
# Test 2: depth control, memorization, probabilities
deep = DecisionTreeClassifier().fit(X_train, y_train)
train_acc = (deep.predict(X_train) == y_train).float().mean().item()
assert train_acc == 1.0, f"an unlimited-depth tree should fit the training set, got {train_acc:.1%}"

for d in (1, 2, 3, 4):
    tree = DecisionTreeClassifier(max_depth=d).fit(X_train, y_train)
    assert tree.get_depth() == d, f"max_depth={d}: expected depth {d}, got {tree.get_depth()}"

P = deep.predict_proba(X_test)
assert P.shape == (len(X_test), 3), P.shape
assert torch.allclose(P.sum(1), torch.ones(len(X_test))), "rows of predict_proba must sum to 1"
assert torch.equal(deep.predict(X_test), P.argmax(1))

shallow = DecisionTreeClassifier(max_depth=3).fit(X_train, y_train)
test_acc = (shallow.predict(X_test) == y_test).float().mean().item()
print(f"max_depth=None: depth {deep.get_depth()}, train accuracy {train_acc:.1%}")
print(f"max_depth=3   : test accuracy {test_acc:.1%}")
assert test_acc > 0.6, f"test accuracy {test_acc:.1%} is unexpectedly low"
print("Test 2 passed")

In [ ]:
# Test 3: edge cases and XOR
pure = DecisionTreeClassifier().fit(X_train, torch.full((len(X_train),), 2))
assert pure.get_depth() == 0, "a pure node must stay a leaf"
assert torch.allclose(pure.predict_proba(X_test[:1]), torch.tensor([[0.0, 0.0, 1.0]]))

const = DecisionTreeClassifier().fit(torch.ones(6, 2), torch.tensor([0, 1, 1, 0, 1, 1]))
assert const.get_depth() == 0, "identical rows: there is no threshold to split on"
assert torch.allclose(const.predict_proba(torch.ones(1, 2)), torch.tensor([[1 / 3, 2 / 3]]))

# XOR: four corner clusters labelled (x0 > 0) xor (x1 > 0)
g = torch.Generator().manual_seed(3)
corners = [((-1.0, -1.0), 0, 16), ((1.0, 1.0), 0, 8), ((-1.0, 1.0), 1, 4), ((1.0, -1.0), 1, 12)]
X_xor = torch.cat([torch.tensor(c) + 0.2 * torch.randn(n, 2, generator=g) for c, _, n in corners])
y_xor = torch.cat([torch.full((n,), label) for _, label, n in corners])
acc = {d: (DecisionTreeClassifier(max_depth=d).fit(X_xor, y_xor).predict(X_xor) == y_xor).float().mean().item()
       for d in (1, 2)}
print(f"XOR accuracy: depth 1 = {acc[1]:.1%}, depth 2 = {acc[2]:.1%}")
assert acc[1] <= 0.75 and acc[2] == 1.0
print("Test 3 passed")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("decision-tree", DecisionTreeClassifier=DecisionTreeClassifier)